# Week 4 · Class 1 — Constitution RAG Pipeline

Build a **RAG** chatbot over the **Constitution of Nepal** with LangChain:

**PDF → document loader → chunks → MiniLM embeddings → in-memory Qdrant → Groq**

## Learning goals

1. Load a PDF with a LangChain document loader (and know other loaders exist)
2. Chunk, embed with `all-MiniLM-L6-v2`, store in **in-memory Qdrant**
3. Answer with a grounded Groq prompt that cites retrieved passages

## Before you start

- Groq key: [console.groq.com](https://console.groq.com)
- PDF: `week-4/data/nepal-constitution.pdf` (placeholder ships in the repo — replace with the real file)

## Section 1 — Install packages

In [ ]:
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface qdrant-client pypdf sentence-transformers

## Section 2 — Groq API key

In [ ]:
import os
import getpass

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    if not os.environ.get("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

print("Key loaded:", bool(os.environ.get("GROQ_API_KEY")))

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
print(llm.invoke("Say 'ready' and nothing else.").content)

## Section 3 — Document loaders (the idea)

LangChain **document loaders** turn files/URLs into `Document` objects (`page_content` + `metadata`).

| Loader | Typical use |
|--------|-------------|
| `PyPDFLoader` / `PyMuPDFLoader` | PDF files |
| `TextLoader` | Plain `.txt` |
| `CSVLoader` | Tabular rows as documents |
| `WebBaseLoader` | Web pages |
| `DirectoryLoader` | Many files in a folder |

Today we use `PyPDFLoader`. The rest of the RAG pipeline stays the same if you swap loaders later.

## Section 4 — Find the PDF

In [ ]:
from pathlib import Path

CANDIDATES = [
    Path("nepal-constitution.pdf"),
    Path("data/nepal-constitution.pdf"),
    Path("../data/nepal-constitution.pdf"),
    Path("/content/nepal-constitution.pdf"),
    Path("/content/data/nepal-constitution.pdf"),
]

PDF_PATH = next((p for p in CANDIDATES if p.exists()), None)
if PDF_PATH is None:
    raise FileNotFoundError(
        "Could not find nepal-constitution.pdf. "
        "Upload it in Colab or place it at week-4/data/nepal-constitution.pdf"
    )
print("Using PDF:", PDF_PATH.resolve())

## Section 5 — Load with `PyPDFLoader`

Each PDF page becomes one LangChain `Document`.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()
print(f"Loaded {len(pages)} page Document(s)")
print("metadata:", pages[0].metadata)
print("--- preview ---")
print(pages[0].page_content[:400])

## Section 6 — Chunk with `RecursiveCharacterTextSplitter`

Defaults: `chunk_size=1000`, `chunk_overlap=200` — good starting point for legal prose.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
chunks = splitter.split_documents(pages)
print(f"Pages {len(pages)} → chunks {len(chunks)}")
print(chunks[0].page_content[:300])

## Section 7 — Free embeddings (`all-MiniLM-L6-v2`)

384-dimensional vectors via LangChain's HuggingFace wrapper (same model family as Week 3).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
probe = embeddings.embed_query("right to equality")
print("dims:", len(probe), "head:", [round(x, 4) for x in probe[:5]])

## Section 8 — Store in **in-memory Qdrant**

`QdrantClient(":memory:")` needs no server. Payloads keep `text`, `source`, and `page`.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "nepal_constitution"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

texts = [c.page_content for c in chunks]
vectors = embeddings.embed_documents(texts)
points = [
    PointStruct(
        id=i,
        vector=vectors[i],
        payload={
            "text": chunks[i].page_content,
            "source": str(chunks[i].metadata.get("source", PDF_PATH.name)),
            "page": chunks[i].metadata.get("page"),
        },
    )
    for i in range(len(chunks))
]
client.upsert(collection_name=COLLECTION, points=points)
print("Points:", client.count(COLLECTION).count)

## Section 9 — Retrieve by meaning

Ask a paraphrase (not a keyword copy-paste) and inspect scores + pages.

In [ ]:
def retrieve(question: str, k: int = 4, score_threshold: float = 0.25):
    qvec = embeddings.embed_query(question)
    hits = client.query_points(
        collection_name=COLLECTION,
        query=qvec,
        limit=k,
    ).points
    return [
        {
            "text": h.payload["text"],
            "source": h.payload.get("source"),
            "page": h.payload.get("page"),
            "score": float(h.score),
        }
        for h in hits
        if h.score >= score_threshold
    ]

hits = retrieve("Are all citizens equal before the law?")
for h in hits:
    print(f"score={h['score']:.3f} page={h['page']}")
    print(h["text"][:220].replace("\n", " "))
    print("---")

## Section 10 — Grounded generation with Groq (LCEL)

Same idea as Week 3: **retrieve → stuff context into a strict prompt → LLM**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

docs_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about the Constitution of Nepal. "
     "Use ONLY the retrieved context below. "
     "Cite pages like [page 3] when available. "
     "If the context is insufficient, say you do not find that in the constitution."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

def format_hits(hits):
    blocks = []
    for h in hits:
        tag = f"page {h['page']}" if h.get("page") is not None else h.get("source", "doc")
        blocks.append(f"[{tag} | score={h['score']:.3f}]\n{h['text']}")
    return "\n\n".join(blocks)

def ask(question: str, k: int = 4, score_threshold: float = 0.25):
    hits = retrieve(question, k=k, score_threshold=score_threshold)
    if not hits:
        return "I could not find that in the constitution (no chunk passed the score threshold)."
    result = (docs_prompt | llm).invoke({
        "question": question,
        "context": format_hits(hits),
    })
    return result.content

print(ask("What does the constitution say about equality?"))
print()
print(ask("Who won the 2022 FIFA World Cup?"))  # honest miss expected

### Mini-exercise (5 min)

1. Ask a rights-related question you expect the constitution to cover.
2. Print `retrieve(...)` for that question — do the pages look right?
3. Raise `score_threshold` to `0.55` and see when answers start failing closed.

In [ ]:
# TODO: your question here
# print(retrieve("..."))
# print(ask("..."))

## Section 11 — Checklist & homework

**Checklist**

- [ ] PDF loads via `PyPDFLoader`
- [ ] Chunks live in in-memory Qdrant with page metadata
- [ ] `ask()` cites context and refuses off-topic questions

**Homework**

1. Replace the placeholder PDF with the full Constitution of Nepal
2. Try `chunk_size` 500 vs 1500 — how do answers change?
3. Bring 5 on-constitution + 5 off-constitution questions to Class 2

**Next class:** wrap this pipeline in **Gradio** (chat + sources + k / threshold sliders).